<a id="unit-u.d5d45f188a484784b9963c49a6459e1a"></a>

<div>
  <div>07-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Ағымдағы күйді қолдау және деректерді қауіпсіз жою</div>
  <p>Реттелген күй өзгерістерін қолданыңыз, қозғалмаған бағандарды сақтаңыз, таңдалған жолдарды түзетіңіз және жойылатын дерек көлеміне сай жою әдісін таңдаңыз.</p>
  <span>Unique Key · Upsert · Sequence бағаны · Ішінара баған жаңарту · UPDATE · MERGE INTO · DELETE · Delete Sign · TRUNCATE · INSERT OVERWRITE</span>
</div>

**7-модуль: Apache Doris жүйесінде деректерді жаңарту және жою** · [Сабақ](course.md) · [Тест](quiz07_state_changes_and_deletion.ipynb)

Шамамен 25 минут жоспарлаңыз.

Бұл зертханалық жұмыс шағын дербес тапсырыс кестелерін пайдаланады. `events`, `events_modelled` немесе `dim_products` өзгертпейді. Әр жазу бөлімі өзгеріс қолданбас бұрын өзінің бақыланатын бастапқы күйін қалпына келтіреді, сондықтан бөлімді жол жинамай қайта орындай аласыз.

Амалды өзгеріс шартына қарай таңдаңыз:

| Өзгеріс шарты | Doris амалы |
|---|---|
| Әр бизнес кілт үшін толық кіріс күйі | Жүктеу негізіндегі толық жолдық upsert |
| Тек таңдалған мән бағандары өзгерген | Ішінара баған жаңарту |
| Предикат таңдаған сирек түзету | SQL `UPDATE` |
| Дайындалған қатынаста шартты енгізу, жаңарту және жою бар | `MERGE INTO` |
| Предикат таңдаған жолдарды жою | SQL `DELETE` |
| Топтама немесе CDC ағыны жойылған бастапқы кілттерді алып келеді | Delete Sign |
| Тұтас кестені немесе Partition жою | `TRUNCATE` |
| Кесте немесе Partition дерексіз аралықсыз қайта құру | `INSERT OVERWRITE` |

<a id="unit-u.9dbadab018b34f60992f4ca7b89af7cf"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі ұяшықты 1-бөлімге дейін орындаңыз. Ол ортақ курс көмекшісін жүктеп, кейінгі әр ұяшық пайдаланатын `lab` нысанын жасайды. Jupyter ядросын қайта іске қосқаннан кейін оны қайта орындаңыз. Ол Docker іске қоспайды және Doris деректерін өзгертпейді.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


<a id="unit-u.d61c79b35dbf4b69910cec2939032003"></a>

<a id="1-keep-one-current-row-per-order-as-changes-arrive"></a>

## 1. Өзгерістер келгенде әр тапсырысқа бір ағымдағы жол сақтау

Оқиға тарихы кестесі әр оқиғаны сақтайды, бірақ `order_state` басқа сұраққа жауап береді: әр тапсырыстың ағымдағы күйі қандай? Оның түйіршіктілігі әр `order_id` үшін бір логикалық жол, сондықтан Unique Key моделі сай.

`updated_at` — Sequence бағаны. Бірнеше жазу сол `order_id` мәнін алып келсе, үлкенірек Sequence мәні бар жол көрінетін қалады. Басқа бағандар — сол күйді сипаттайтын мән бағандары.

Upsert жаңарту мен енгізу мағынасын бір жазуда біріктіреді: кілт бар болса, Doris оның көрінетін күйін ауыстырады; кілт жоқ болса, жаңа логикалық жол енгізеді. Үш бастапқы тапсырысты көрсеткеннен кейін сол ұяшық бар `1001` тапсырысына толық `shipped` күйін жазып, жаңа `1004` тапсырысын енгізеді. `1001` тапсырысының кешіккен `paid` оқиғасы соңынан келеді, бірақ `updated_at` мәні көрінетін `shipped` күйінен бес минут ескі.


In [ ]:
lab.start_container()
lab.connect()
lab.execute("USE doris_course")
lab.execute("DROP TABLE IF EXISTS order_state")
lab.execute("""
CREATE TABLE order_state (
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'created',
    amount DECIMAL(12, 2) NOT NULL DEFAULT '0.00',
    shipping_region VARCHAR(16) NOT NULL DEFAULT 'UNASSIGNED',
    updated_at DATETIME NOT NULL
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "1",
    "function_column.sequence_col" = "updated_at"
)
""")

lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'created',    99.90, 'CN-EAST',  '2026-09-11 09:00:00'),
    (1002, 502, 'paid',      120.00, 'CN-SOUTH', '2026-09-11 09:05:00'),
    (1003, 503, 'processing', 88.00, 'CN-WEST',  '2026-09-11 09:10:00')
""", title="Write the initial order states")

lab.sql("SELECT * FROM order_state ORDER BY order_id", title="Initial current-state rows")

lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'shipped', 99.90, 'CN-EAST',  '2026-09-11 10:10:00'),
    (1004, 504, 'created', 42.00, 'CN-NORTH', '2026-09-11 10:00:00')
""", title="Upsert a newer state and a new order")

lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'paid', 99.90, 'CN-EAST', '2026-09-11 10:05:00')
""", title="Send a delayed older state")

lab.sql("""
SELECT order_id, status, amount, updated_at
FROM order_state
WHERE order_id IN (1001, 1004)
ORDER BY order_id
""", title="Visible states after ordered upserts");

<a id="unit-u.1e987cbbebe2409999380d7a66f15117"></a>

**Күтілетін нәтиже:** бірінші нәтижеде әр `order_id` үшін біреуден үш бастапқы жол бар. Соңғы нәтиже `1001` тапсырысын `10:10` уақытында `shipped` деп көрсетеді, `paid` жолы кейін келгеніне қарамастан, өйткені `10:05` Sequence мәні кіші. `1004` жаңа жол болып көрінеді және кестеде алты тарихи нұсқа емес, төрт логикалық жол бар.

Бұл upsert екі тармағын да көрсетеді: `1001` бар кілтті жаңартады, ал `1004` жаңа кілт енгізеді. Doris 4.x ішінде Unique Key моделі әдепкі Merge-on-Write пайдаланады. Құжатталған механизм бойынша Doris қабылданған алмастыруды жаңа Rowset ішіне жазып, ауыстырылған жолды delete bitmap ішінде тіркейді; кейінгі сұраулар сол ескі жол нұсқасын өткізіп жібереді, ал физикалық тазалау кейін Compaction кезінде болады. Жоғарыдағы көрінетін жолдар сол ішкі құрылымдарды тікелей емес, жеңген күйді көрсетеді. Ұяшықты қайта орындау сол өзгерістерді қолданбас бұрын кестені жойып, қайта құрады.


<a id="unit-u.291e305a0671421ea1841cde6f200b6a"></a>

<a id="2-choose-whether-omitted-columns-use-defaults-or-retain-existing-values"></a>

## 2. Түсірілген бағандардың әдепкілерді қолданатынын немесе бар мәндерді сақтайтынын таңдау

Екі сынақ тапсырысы да сома мен жеткізу аймағынан басталады. `1002` тапсырысының толық жолдық upsert амалы сол бағандарды түсіріп қалдырады, сондықтан Doris оларды жарияланған әдепкілерімен толтырады. `1003` тапсырысының ішінара баған жаңартуы тек `status` және `updated_at` өзгертеді; Doris түсірілген бағандардың бар мәндерін сақтайды.

`INSERT INTO` үшін `enable_unique_key_partial_update` сеанс айнымалысы осы айырманы айқын етеді. Stream Load және басқа қабылдау жолдары өздерінің тиісті `partial_columns` баптауын пайдаланады.


In [ ]:
lab.execute("TRUNCATE TABLE order_state")
lab.insert("""
INSERT INTO order_state VALUES
    (1002, 502, 'paid', 120.00, 'CN-SOUTH', '2026-09-11 09:05:00'),
    (1003, 503, 'paid',  88.00, 'CN-WEST',  '2026-09-11 09:10:00')
""", title="Restore two complete order states")

lab.execute("SET enable_unique_key_partial_update = false")
lab.insert("""
INSERT INTO order_state (order_id, customer_id, status, updated_at)
VALUES (1002, 502, 'shipped', '2026-09-11 10:20:00')
""", title="Apply a full-row upsert with omitted columns")

lab.execute("SET enable_unique_key_partial_update = true")
lab.insert("""
INSERT INTO order_state (order_id, status, updated_at)
VALUES (1003, 'shipped', '2026-09-11 10:20:00')
""", title="Update only the changed columns")
lab.execute("SET enable_unique_key_partial_update = false")

lab.sql("""
SELECT
    order_id, status, amount, shipping_region,
    IF(order_id = 1002, 'full-row upsert', 'partial column update') AS update_path
FROM order_state
ORDER BY order_id
""", title="Full-row and partial-update results");


<a id="unit-u.9328d04b72d84bf7bd0e8e384f536cf7"></a>

**Күтілетін нәтиже:** екі тапсырыс та `shipped` болады. Толық жолдық upsert баған әдепкілерін пайдаланғандықтан, `1002` тапсырысында енді `amount = 0.00` және `shipping_region = UNASSIGNED`. Ішінара баған жаңарту алмаған мәндерін сақтағандықтан, `1003` тапсырысы `88.00` және `CN-WEST` сақтайды. Өндіруші кең ағымдағы күй жолының тек бір бөлігіне иелік етсе, ішінара жаңарту пайдаланыңыз.


<a id="unit-u.e2313fb398f449aaa9a2113023fc0e00"></a>

<a id="3-correct-a-small-selected-set-with-sql-update"></a>

## 3. SQL UPDATE арқылы шағын таңдалған жиынды түзету

SQL `UPDATE` предикатпен көрсетілген сирек түзету үшін пайдалы. Doris `WHERE` таңдаған жолдарды сканерлеп, жаңартылған жол нұсқаларын Unique Key кестесіне қайта жазады. Бұл қолданбаның жеке жазуларының тығыз циклі үшін таңдаулы жол емес.

`UPDATE` тек мән бағандарын өзгерте алады. `order_id` өзгерту бизнес кілтті өзгертер еді, сондықтан ол ескі кілтті жою және жаңа кілт енгізу ретінде көрсетілуі тиіс.

| Жаңарту жолы | Жолдар қалай таңдалады | Сай бастапқы тәсіл | Қозғалмаған мән бағандары |
|---|---|---|---|
| Толық жолдық upsert | Кіріс кілт мәндері | Толық күйлер немесе CDC жазбаларының топтамалары | Берілген мәндермен немесе жарияланған әдепкілермен ауысады |
| Ішінара баған жаңарту | Кіріс кілт мәндері | Тек таңдалған бағандары бар өзгеріс топтамалары | Бар жолдан сақталады |
| SQL `UPDATE` | SQL `WHERE` предикаты | Сирек шартты түзету | `SET` ішінде аталмаса, сақталады |


In [ ]:
lab.execute("TRUNCATE TABLE order_state")
lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'shipped', 99.90, 'CN-EAST', '2026-09-11 10:10:00'),
    (1002, 502, 'paid',   120.00, 'CN-SOUTH', '2026-09-11 09:05:00')
""", title="Restore the rows used by the correction")

lab.sql("""
SELECT order_id, status, amount, shipping_region, updated_at
FROM order_state
ORDER BY order_id
""", title="Order states before SQL UPDATE")

lab.insert("""
UPDATE order_state
SET shipping_region = 'CN-NORTH'
WHERE order_id = 1001
""", title="Correct one shipping region")

lab.sql("""
SELECT order_id, status, amount, shipping_region, updated_at
FROM order_state
ORDER BY order_id
""", title="Order states after the correction");


<a id="unit-u.c8106fcf604f4f0c9d5c65736b5536f8"></a>

**Күтілетін нәтиже:** екі нәтиже кестесін салыстырыңыз. `UPDATE` алдында `1001` тапсырысында `shipping_region = CN-EAST`, кейін `CN-NORTH` болады. Оның күйі, сомасы және уақыт белгісі өзгермейді, ал `1002` тапсырысының барлық мәніне әсер етпейді. `WHERE` предикаты жолды таңдады, ал `SET` өзгертілетін жалғыз мән бағанын атады.

1–3-бөлімдер бірге үш жолды ажыратады: толық жолдық upsert толық кіріс күйлерін жазады, ішінара баған жаңарту түсірілген мәндерді сақтайды, ал SQL `UPDATE` сирек предикаттық түзету орындайды. Үлкен көлемді өзгеріс ағындары әдетте көптеген жеке SQL `UPDATE` операторларының орнына жүктеу негізіндегі upsert топтамаларын пайдалануы тиіс.


<a id="unit-u.a7a6aa58615e433298c3ecfe6cb07725"></a>

<a id="4-apply-conditional-changes-from-a-staged-relation-with-merge-into"></a>

## 4. Дайындалған қатынастан MERGE INTO арқылы шартты өзгерістер қолдану

Жергілікті аралық кесте үш бастапқы өзгерісті қамтиды: бар `4001` тапсырысын жаңарту, бар `4002` тапсырысын жою және жаңа `4004` тапсырысын енгізу. Бір бастапқы қатынас сәйкестік күйі мен `change_type` бойынша әртүрлі әрекет қажет еткендіктен, `MERGE INTO` сай келеді.

Мақсат Unique Key кестесі болуы тиіс. Merge орындалмас бұрын бастапқы жол санын бірегей `order_id` санымен салыстырыңыз. Doris `MERGE INTO` ішінде қайталанатын Join жолдарын анықтамайды; бірнеше бастапқы жолдың сол мақсатты жолға әрекет етуі анықталмаған нәтиже беруі мүмкін. Өндірістік құбыр беделді бастапқы реттілікпен қайталанудан тазартуы немесе топтаманы қабылдамауы тиіс.


In [ ]:
lab.execute("DROP TABLE IF EXISTS order_merge_changes")
lab.execute("DROP TABLE IF EXISTS order_merge_target")
lab.execute("""
CREATE TABLE order_merge_target (
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    amount DECIMAL(12, 2) NOT NULL,
    shipping_region VARCHAR(16) NOT NULL,
    updated_at DATETIME NOT NULL
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.execute("""
CREATE TABLE order_merge_changes (
    order_id BIGINT NOT NULL,
    change_type VARCHAR(10) NOT NULL,
    status VARCHAR(20) NULL,
    amount DECIMAL(12, 2) NULL,
    shipping_region VARCHAR(16) NULL,
    updated_at DATETIME NULL
)
DUPLICATE KEY(order_id, change_type)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO order_merge_target VALUES
    (4001, 'paid',       99.90, 'CN-EAST',  '2026-09-11 09:00:00'),
    (4002, 'processing',120.00, 'CN-SOUTH', '2026-09-11 09:05:00'),
    (4003, 'paid',       75.00, 'CN-WEST',  '2026-09-11 09:10:00')
""", title="Write the MERGE target state")
lab.insert("""
INSERT INTO order_merge_changes VALUES
    (4001, 'UPSERT', 'shipped', 99.90, 'CN-EAST',  '2026-09-11 10:00:00'),
    (4002, 'DELETE', NULL,      NULL,  NULL,       NULL),
    (4004, 'UPSERT', 'created', 50.00, 'CN-NORTH', '2026-09-11 10:05:00')
""", title="Stage update, delete, and insert changes")
lab.sql("""
SELECT
    COUNT(*) AS source_rows,
    COUNT(DISTINCT order_id) AS distinct_order_keys,
    IF(COUNT(*) = COUNT(DISTINCT order_id), 'ready', 'reject') AS merge_readiness
FROM order_merge_changes
""", title="Validate one source row per target Key")
lab.insert("""
MERGE INTO order_merge_target AS t
USING order_merge_changes AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.change_type = 'DELETE' THEN DELETE
WHEN MATCHED AND s.change_type = 'UPSERT' THEN UPDATE SET
    status = s.status,
    amount = s.amount,
    shipping_region = s.shipping_region,
    updated_at = s.updated_at
WHEN NOT MATCHED AND s.change_type = 'UPSERT' THEN INSERT
    (order_id, status, amount, shipping_region, updated_at)
VALUES
    (s.order_id, s.status, s.amount, s.shipping_region, s.updated_at)
""", title="Merge the staged change set")
lab.sql("SELECT * FROM order_merge_target ORDER BY order_id", title="Current state after MERGE INTO");


<a id="unit-u.369c55278afd4be7aaacaef926c9b5c8"></a>

**Күтілетін нәтиже:** тексеру жолы `source_rows = 3`, `distinct_order_keys = 3` және `merge_readiness = ready` деп хабарлайды. Merge кейін:

- `4001` `10:00` уақытында `shipped`, өйткені сәйкес UPSERT `UPDATE` әрекетін таңдады;
- `4002` жоқ, өйткені сәйкес DELETE `DELETE` әрекетін таңдады;
- `4003` өзгермейді, өйткені ешбір бастапқы жол сәйкес келмеді;
- `4004` `10:05` уақытында `created`, өйткені сәйкес емес UPSERT `INSERT` әрекетін таңдады.

Сондықтан соңғы мақсатта үш көрінетін жол бар: `4001`, `4003` және `4004`. Бұл — осы бірегей үш жолдық дереккөз үшін әрекеттер мен нәтиженің дәлелі. Дереккөз шарты әдейі қабылдамайтын қайталанатын бастапқы сәйкестіктердің әрекетін анықтамайды.


<a id="unit-u.fceceddb05324d709f0d8d76a96c5d57"></a>

<a id="5-remove-rows-by-predicate-or-incoming-key"></a>

## 5. Жолдарды предикат немесе кіріс кілт арқылы жою

Кестеде сақталатын бір жол және жойылатын екі жол бар. SQL `DELETE` предикатпен сынақ жолын таңдайды. Екінші жою кіріс кілт үшін `__DORIS_DELETE_SIGN__ = 1` жазады; бұл топтамалық жоюды немесе CDC дереккөзінің жою оқиғасын модельдейді.

Қалыпты сұраулар жойылған жолдарды жасырады. `show_hidden_columns` қосу осы тексеру үшін жою белгісін көрінетін етеді; қолданба сұраулары оны әдетте өшірулі қалдырады. Мұнда механизмді бір жолмен көрсету үшін `INSERT INTO` қолданылады. Өндірістік CDC және үлкен топтамалар әдетте сол белгіні жүктеу жолы арқылы жеткізеді.

Екі Tablet көрінісі жоюлардың алдындағы және кейінгі сақтау дәлелін береді. `Version` — ең соңғы жарияланған Tablet нұсқасы; транзакция жарияланған бойда FE оны жаңартады. `SHOW TABLETS` ішінде `VersionCount` және `RowCount` та бар, бірақ әр BE оларды шамамен минутына бір рет хабарлайды, сондықтан жазудан кейін бірден олар әлі `-1` және `0` көрсетеді. Сондықтан көріністер тек `TabletId` және `Version` көрсетеді.

In [ ]:
lab.execute("DROP TABLE IF EXISTS order_deletions")
lab.execute("""
CREATE TABLE order_deletions (
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'unknown',
    updated_at DATETIME NOT NULL DEFAULT '1970-01-01 00:00:00'
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO order_deletions VALUES
    (2001, 'paid',      '2026-09-11 09:00:00'),
    (2002, 'cancelled', '2026-09-11 09:05:00'),
    (2003, 'test',      '2026-09-11 09:10:00')
""", title="Write the controlled deletion cases")

lab.sql("""
SELECT order_id, status, updated_at, COUNT(*) OVER() AS visible_row_count
FROM order_deletions
ORDER BY order_id
""", title="Visible rows before deletion")
lab.sql(
    "SHOW TABLETS FROM order_deletions",
    title="Tablet version before deletion",
    columns=["TabletId", "Version"],
)

lab.execute("DELETE FROM order_deletions WHERE order_id = 2003")
lab.execute("SET show_hidden_columns = true")
lab.insert("""
INSERT INTO order_deletions (order_id, __DORIS_DELETE_SIGN__)
VALUES (2002, 1)
""", title="Write an incoming Delete Sign")
lab.execute("SET show_hidden_columns = false")

lab.sql("""
SELECT order_id, status, updated_at, COUNT(*) OVER() AS visible_row_count
FROM order_deletions
ORDER BY order_id
""", title="Visible rows after deletion")
lab.sql(
    "SHOW TABLETS FROM order_deletions",
    title="Tablet version after deletion",
    columns=["TabletId", "Version"],
)

lab.execute("SET show_hidden_columns = true")
lab.sql("""
SELECT order_id, status, updated_at, __DORIS_DELETE_SIGN__
FROM order_deletions
ORDER BY order_id
""", title="Stored delete markers")
lab.execute("SET show_hidden_columns = false");

<a id="unit-u.ed82a2a5bccc4b539af6f51956e38484"></a>

**Күтілетін нәтиже:** жоюға дейін үш жол көрінеді және әр жол `visible_row_count = 3` көрсетеді; олардың `status` және `updated_at` мәндері бастапқы күйді көрсетеді. Екі жоюдан кейін қалыпты сұрау тек `2001` тапсырысын `status = paid`, `updated_at = 2026-09-11 09:00:00` және `visible_row_count = 1` мәндерімен қайтарады. Екінші Tablet көрінісі біріншіден үлкенірек `Version` көрсетеді: предикаттық `DELETE` және Delete Sign жазуы әрқайсысы бір жаңа нұсқа жариялады.

Қалыпты сұрау нәтижесі және кейінгі Tablet `Version` — тікелей бақыланған дәлел: транзакция логикалық көрінуді өзгертіп, жаңа сақтау нұсқаларын жариялады. Doris Merge-on-Write механизмі бойынша ескі жол нұсқалары delete bitmap ішінде көрінбейтін болып белгіленіп, кейінгі Compaction арқылы тазартылады. Сондықтан өзгерген сұрау нәтижесі логикалық жоюды білдіреді; физикалық сақтау бірден босатылды дегенді білдірмейді.

Жасырын бағандар қосылғанда, `2002` және `2003` тапсырыстарында `__DORIS_DELETE_SIGN__ = 1` бар: `2002` белгіні тікелей алды, ал предикаттық `DELETE` `2003` логикалық жойылуын жасады. Жою жазбасы бұрынғы бизнес мәндердің орнына әдепкілер қамтуы мүмкін, сондықтан бұл көріністі жойылған күйді қалпына келтіруге емес, белгіні тексеруге пайдаланыңыз. Delete Sign тек Unique Key кестелерінде бар және көптеген жойылған кілт кіріс дерегі ретінде келгенде пайдалы. SQL шарты жойылатын жолдарды анықтаса, предикаттық `DELETE` — жалпы таңдау. `__DORIS_DELETE_SIGN__` — кіріске арналған жасырын баған; ішкі әр Rowset delete bitmap құрылымы емес.

<a id="unit-u.cef2381e1c444a148b7d50c9731b6b17"></a>

<a id="6-match-whole-partition-maintenance-to-a-lifecycle-operation"></a>

## 6. Тұтас Partition күтімін өмірлік цикл амалына сәйкестендіру

Partition ішіндегі әр жолды бір-бірден жою қажетсіз жұмыс жасайды. Келесі кесте екі тапсырыс күнін Partition бөлімдеріне бөледі. `TRUNCATE PARTITION` мерзімі өткен күнді метадеректер арқылы жояды, ал `INSERT OVERWRITE` алмастыру мазмұнын құрып, таңдалған Partition ішіне атомарлы ауыстырады.

Сұраулар әр өмірлік цикл амалынан кейінгі күйді көрсетеді. Бұл амалдар тек шағын `order_lifecycle` кестесінде орындалады.


In [ ]:
lab.execute("DROP TABLE IF EXISTS order_lifecycle")
lab.execute("""
CREATE TABLE order_lifecycle (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p20260910 VALUES LESS THAN ('2026-09-11'),
    PARTITION p20260911 VALUES LESS THAN ('2026-09-12'),
    PARTITION pmax VALUES LESS THAN MAXVALUE
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO order_lifecycle VALUES
    ('2026-09-10', 3001, 'expired'),
    ('2026-09-10', 3002, 'expired'),
    ('2026-09-11', 3003, 'paid'),
    ('2026-09-11', 3004, 'paid')
""", title="Write two daily Partitions")

lab.sql("""
SELECT order_date, COUNT(*) AS order_rows
FROM order_lifecycle
GROUP BY order_date
ORDER BY order_date
""", title="Rows before lifecycle maintenance")

lab.execute("TRUNCATE TABLE order_lifecycle PARTITION(p20260910)")
lab.sql("""
SELECT order_date, COUNT(*) AS order_rows
FROM order_lifecycle
GROUP BY order_date
ORDER BY order_date
""", title="Rows after TRUNCATE PARTITION")

lab.insert("""
INSERT OVERWRITE TABLE order_lifecycle PARTITION(p20260911) VALUES
    ('2026-09-11', 3003, 'shipped'),
    ('2026-09-11', 3005, 'created')
""", title="Atomically replace the active Partition")
lab.sql("SELECT * FROM order_lifecycle ORDER BY order_id", title="Rows after INSERT OVERWRITE");


<a id="unit-u.af9283640da54b6aa999f0c385fa5b38"></a>

**Күтілетін нәтиже:** бірінші нәтижеде әр күнге екі жол бар. `TRUNCATE PARTITION` кейін тек екі `2026-09-11` жолы қалады. `INSERT OVERWRITE` кейін сол Partition ішінде түзетілген `shipped` күйімен `3003` тапсырысы және жаңа `3005` тапсырысы бар; ескі `3004` енді жоқ. Overwrite транзакциясы бекітілгенше оқу сұраулары ескі Partition мазмұнын көруді жалғастыра алады, бұл жоюдан кейін қайта жүктеу аралығын болдырмайды.


<a id="unit-u.2e934d22794e4573b5c9f44392ee3c65"></a>

<a id="7-practice-patch-orders-through-stream-load"></a>

## 7. Жаттығу: тапсырыстарды Stream Load арқылы ішінара жаңарту

2-бөлім бір тапсырысты `INSERT` және сеанс айнымалысы арқылы өзгертті. Логистика жүйесі көбіне файл жіберер еді. Бағандары `order_id`, `status`, `amount DECIMAL(10,2) NOT NULL DEFAULT '0.00'` және `shipping_region VARCHAR(16) NOT NULL DEFAULT 'UNASSIGNED'` болатын шағын Unique Key `order_patch_practice` кестесін жасап, екі тапсырыс енгізіңіз: `(5001, 'paid', 120.00, 'CN-EAST')` және `(5002, 'paid', 80.00, 'CN-WEST')`.

Содан кейін мына екі жолдық файлды Stream Load арқылы `partial_columns:true` және `columns:order_id,status` тақырыптарымен жіберіңіз:

```text
5001,shipped
5003,created
```

Орындамас бұрын одан кейінгі үш тапсырыстың бәрін болжаңыз: күйлері, сомалары және аймақтары. Аяқтағаннан кейін жаттығу кестесін жойыңыз.

In [ ]:
# Your code here

<a id="unit-u.9be0754fde48443a83484f1342886a87"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
lab.execute("DROP TABLE IF EXISTS order_patch_practice")
lab.execute("""
CREATE TABLE order_patch_practice (
    order_id BIGINT NOT NULL,
    status VARCHAR(16) NOT NULL,
    amount DECIMAL(10,2) NOT NULL DEFAULT "0.00",
    shipping_region VARCHAR(16) NOT NULL DEFAULT "UNASSIGNED"
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.execute("""
INSERT INTO order_patch_practice VALUES
    (5001, 'paid', 120.00, 'CN-EAST'),
    (5002, 'paid', 80.00, 'CN-WEST')
""")

lab.stream_load(f"""
printf '5001,shipped\\n5003,created\\n' |
curl --silent --show-error --location-trusted -u root: \\
  -H "Expect:100-continue" \\
  -H "partial_columns:true" \\
  -H "column_separator:," \\
  -H "columns:order_id,status" \\
  -T - \\
  -X PUT {lab.fe_http_url}/api/doris_course/order_patch_practice/_stream_load
""", title="Partial update through Stream Load", expected_status="Success",
    expected_counts={"NumberLoadedRows": 2}, show_json=False)

patched = lab.sql("SELECT * FROM order_patch_practice ORDER BY order_id",
                  title="Orders after the patch")
assert patched["order_id"].astype(int).tolist() == [5001, 5002, 5003]
assert patched["status"].tolist() == ["shipped", "paid", "created"]
assert [str(value) for value in patched["amount"]] == ["120.00", "80.00", "0.00"], \
    "Existing orders must retain their amounts; the new order uses its default."
assert patched["shipping_region"].tolist() == ["CN-EAST", "CN-WEST", "UNASSIGNED"], \
    "Existing orders must retain their regions; the new order uses its default."
lab.execute("DROP TABLE order_patch_practice");

<a id="unit-u.bf0657e0c39344918c275f2330fd6060"></a>

**Күтілетін нәтиже**

Жауап 2 жүктелген жолмен `Success` хабарлайды, ал кестеде үш тапсырыс бар:

| order_id | status | amount | shipping_region |
|---:|---|---:|---|
| 5001 | shipped | 120.00 | CN-EAST |
| 5002 | paid | 80.00 | CN-WEST |
| 5003 | created | 0.00 | UNASSIGNED |

5001 тапсырысы тек күйін өзгертті: ішінара жаңарту файлда жоқ бағандарды сақтайды. 5002 файлда жоқ және сол қалпы қалады. 5003 бұрын жоқ еді, сондықтан сақтайтын мәндер болмады: `partial_update_new_key_behavior` әдепкісі `APPEND` болғандықтан, Doris оны енгізіп, `amount` және `shipping_region` бағандарын әдепкілерімен толтырды. `partial_columns:true` болмаса, сол файл толық жолдық upsert болып, 5001 сомасы мен аймағын да әдепкілеріне қайта орнатар еді.

<a id="unit-u.749db1b48b914764b5e73c8e69342df3"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

CPU мен жадты босату үшін осы қосымша ұяшықты орындаңыз. Docker атаулы томдары 7-модуль кестелерін сақтайды.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");


<a id="unit-u.4513082de6b54f0fb5d4a816f33b5dc5"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Басқа модульге өтпес бұрын осы ұяшықты орындаңыз. Ол бар контейнерді іске қосып, FE түйініне қайта қосылады және сақталған ағымдағы күй кестесін тексереді.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container()

lab.connect()
lab.execute("USE doris_course")
lab.sql("SELECT COUNT(*) AS current_orders FROM order_state", title="Recovered current-state table");


<a id="unit-u.4b65ca8241904acdb0fb01be7b4a6d7c"></a>

**Күтілетін нәтиже:** контейнер `healthy` күйіне оралады және `current_orders` 3-бөлімде қалпына келтірілген екі жолды көрсетеді.

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Әр тапсырысқа бір ағымдағы жол модельдедіңіз, ескі күйді қабылдамау үшін Sequence бағанын пайдаландыңыз, толық жолдық және ішінара баған мағынасын ажыраттыңыз, предикаттық түзету қолдандыңыз, дайындалған жаңарту/жою/енгізу жиынын біріктірдіңіз, предикаттық жоюды Delete Sign тәсілімен салыстырдыңыз және тұтас Partition күтімін `TRUNCATE` пен `INSERT OVERWRITE` амалдарына сәйкестендірдіңіз.

Ресми анықтамалар: [Unique Key](https://doris.apache.org/docs/4.x/key-features/unique-key/) · [Unique моделі үшін жүктеу арқылы жаңартулар](https://doris.apache.org/docs/4.x/data-operate/update/update-of-unique-model/) · [UPDATE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-modification/DML/UPDATE/) · [MERGE INTO](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-modification/DML/MERGE-INTO/) · [Жүктеу арқылы топтамалық жою](https://doris.apache.org/docs/4.x/data-operate/delete/batch-delete-manual/) · [Жою амалдарына шолу](https://doris.apache.org/docs/4.x/data-operate/delete/delete-overview/) · [TRUNCATE](https://doris.apache.org/docs/4.x/data-operate/delete/truncate-manual/) · [INSERT OVERWRITE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-modification/DML/INSERT-OVERWRITE/)

<a id="unit-u.d68c4ab9ef40438cb4a116c0e0a0d38c"></a>

<a id="next"></a>

## Келесі қадам

[7-тест: Apache Doris жүйесінде деректерді жаңарту және жою](quiz07_state_changes_and_deletion.ipynb) ашыңыз. Бұл — 2-деңгейдің соңғы зертханалық жұмысы; 3-деңгей [8-модуль: Apache Doris жүйесінде сұрауларды жеделдету](../../level3/module08-query-acceleration/course.md) бөлімінен басталады.
